## Sequential Monte Carlo

Adjustments:
- replace manual logging with arvis inference structure (easier/faster saving in netcdf format, immediate access to plotting routines)
- replace prior class with scipy priors (compatibility with tinyDA code above)
- replace loglikelihood class with tinyDA style loglikelihood
- replace MCMC steps with tinyDA

In [16]:
from os import sys, path, getcwd
sys.path.append(path.dirname(getcwd()))

import functools as ft

import warnings
import matplotlib.cbook
import scipy.stats as stats

from SMC import *

Finished: Wed Jul 30 16:08:34 2025


Start with the example provided before moving on to more complex model

In [51]:
### M:  number of particles/samples for SMC
num_samples = 10

### var: uncertainty variance of measurements/modeling
std_noise = 0.05

### Title for saving the results
run = "acousticgravity"  # or any identifier you want
resultspath = f'RESULTS/SMC_M_{num_samples}_{run}'

In [52]:
# Wrapper for scipy prior
class prior:
    def __init__(self, prior, d = 1):
        self.d = d # dimension of prior/parameter space, here 1
        self.RV = prior # this is specific to this prior

    def rvs(self, M):
        # Generate n samples. The resulting numpy array must be [1:d, 1:M]
        return np.array([self.RV.rvs(size=M) for i in range(self.d)])

    def logpdf(self, particle):
        # Compute logpdf at a point "particle"-
        return np.sum([self.RV.logpdf(particle[i]) for i in range(self.d)])

nx = 1 # 1D test case just to start!
length_scale = 5.0
eps = 1e-6
decay_rate = 10

x = np.linspace(0, nx - 1, nx).reshape(-1, 1)
def rbf_covariance(grid, length_scale, variance=1.0):
    dists = cdist(grid, grid, 'euclidean')
    return variance * np.exp(-0.5 * (dists / length_scale) ** 2)

cov = rbf_covariance(x, length_scale)
cov += eps * np.eye(nx)
mean = np.zeros(nx)
my_prior = prior(stats.multivariate_normal(mean=mean, cov=cov), d=1)

In [53]:
### Set up forward model
def forward_model(x0):
    #Transform x0 to single location in input parameter vector (simplified testcase, to be replaced later)
    # True x0 is 70
    nx = umbridge_model.get_input_sizes(config)[0]
    xvec = np.zeros(nx)
    if 0 <= x0[0] - 10 < nx: xvec[int(x0[0]) - 10] = 1
    if 0 <= x0[0] + 10 < nx: xvec[int(x0[0]) + 10] = 1
    output = levelset_model(xvec)   
    return output

def potential(particle, data, std_noise = 0.05):
    fwd_vals = forward_model(particle)
    #return 1.*(np.linalg.norm(fwd_vals-data)**2/(2.*std_noise**2)) 
    return my_loglike.loglike(fwd_vals)

# Multiprocessing evaluation of previous potential 
def potential_mp(particles,data, std_noise = 0.05):
    pool = mp.Pool()
    potentials = pool.map(ft.partial(potential, data=data, std_noise=std_noise),particles.T)
    pool.close()
    return np.array(potentials)


In [54]:
### SET UP SMC SAMPLER  
SMC_sampler = SMC(my_prior, lambda x: potential_mp(x, data = d_true, std_noise = std_noise), resultspath, d_true, num_samples)
SMC_sampler.smc_tempering() ### perform SMC algorithm

SMC for RESULTS/SMC_M_10_acousticgravity - Started: Wed Jul 30 16:20:34 2025
------ TEMPERING-STEP 0 --------------------------

>> REWEIGHTED with exponent 1.0
>> NOT resampled (ESS 99.99% , exponent = 1)

--- duplicates: 0/10  |  highest count: 1x
Applying MCMC kernel 4 times 

Acceptance ratio: 0.8 >0.3 
  => variance scaling: 1.0->2.0
>> MCMC complete!
--- duplicates: 0/10  |  highest count: 1x
[saved intermediate result]
Tempering finished!
Tempering steps done:  0
